## Open notebook in:
| Colab                                 
:-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nicolepcx/transformers-the-definitive-guide/blob/master/CH09/09_langgraph_multiturn_conversation.ipynb)                                             

# About the Notebook



## 🔁 Interactive LinkedIn Post Generator with Human-in-the-Loop Feedback

This notebook implements a LangGraph workflow that generates LinkedIn posts using an LLM, with a human-in-the-loop feedback mechanism. The flow includes three main components:

- **Model Node**: Uses a language model (Llama 3 via Nebius) to generate a LinkedIn post based on a user-provided topic and optional feedback.
- **Human Node**: Pauses execution to collect user feedback on the generated post. The user can iteratively refine the output or type `"done"` (or `"exit"`, `"quit"`, `"q"`) to finalize it.
- **End Node**: Outputs the final post and feedback summary.

The code uses LangGraph’s `interrupt()` and `and command()` mechanism to handle real-time input, and supports continuous, feedback-driven improvement of the generated content. The flow is structured using a `StateGraph`, and all interactions are managed with a `while True` loop that handles streaming and user input cleanly.

**NOTE:** You can use either Open AI or any other LLM here, I used [Nebius](https://studio.nebius.com/), which offers fairly good prices on some common LLMs.



# Dependencies

In [ ]:
%%capture --no-stderr
%pip install openai==1.61.1 python-dotenv==1.0.1 langchain-openai==0.2.13 langgraph==0.3.31

# API Setup

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# Set up API keys and environment variables
NEBIUS_API_KEY = os.getenv('NEBIUS_API_KEY')

# Uncomment to use OpenAI
#OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

In [ ]:
# Uncomment to use OpenAI
#llm = ChatOpenAI(model="gpt-4o")

# Imports

In [ ]:
from __future__ import annotations

import uuid
from typing import List, TypedDict, Annotated
from operator import add as add_messages

from langgraph.graph import StateGraph, START
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

A `StateGraph` object defines the structure of our chatbot as a "state machine". We'll add `nodes` to represent the llm and functions our chatbot can call and `edges` to specify how the bot should transition between these functions.

# Define Human Node Using `Interrupt()`

# Constants

In [ ]:
llm = ChatOpenAI(
                model="meta-llama/Llama-3.3-70B-Instruct-fast",
                temperature=0,
                max_tokens=None,
                timeout=None,
                max_retries=2,
                api_key=NEBIUS_API_KEY,
                base_url="https://api.studio.nebius.ai/v1/"
            )


In [ ]:


# -------------------------
# State
# -------------------------
class State(TypedDict):
    linkedin_topic: str
    generated_post: Annotated[List[AIMessage], add_messages]
    human_feedback: Annotated[List[str], add_messages]


# -------------------------
# Helpers
# -------------------------
DONE_SET = {"done", "finish", "final", "finalize", "finalise", "quit", "exit", "q"}
OKAY_SET = {"ok", "okay", "k", "looks good", "fine", "good", "ship it"}

def _summarize_feedback(feedback_list: List[str], keep_last: int = 3) -> str:
    """Keep the last few feedback items to avoid prompt bloat."""
    if not feedback_list:
        return "okay"
    trimmed = feedback_list[-keep_last:]
    return " | ".join(trimmed)


# -------------------------
# Nodes
# -------------------------
def model(state: State) -> State:
    """Draft a LinkedIn post using the topic and the most recent human feedback."""
    topic = state["linkedin_topic"]
    feedback_list = state.get("human_feedback", [])
    feedback_summary = _summarize_feedback(feedback_list)

    prompt = f"""
    You are an expert LinkedIn content writer. Create a strong post that:
    - Opens with a clear hook in 1 to 2 short sentences
    - Uses short paragraphs with concrete points and a practical example
    - Avoids fluff and clichés
    - Ends with a simple call to action
    - Keeps a professional but conversational tone

    Topic: {topic}
    Latest feedback to consider: {feedback_summary}
    Return plain text only.
    """.strip()

    response = llm.invoke([
        SystemMessage(content="Write crisp, useful LinkedIn posts with substance and clarity."),
        HumanMessage(content=prompt)
    ])

    generated = response.content
    print("\n[model] Draft ready. Handing over for review...\n")
    print("----- DRAFT START -----")
    print(generated)
    print("------ DRAFT END ------\n")

    return {
        "linkedin_topic": topic,
        "generated_post": [AIMessage(content=generated)],
        "human_feedback": feedback_list,
    }


def human_node(state: State) -> Command:
    """Pause for human feedback. Default to 'okay' if the user presses Enter."""
    latest = state["generated_post"][-1].content if state["generated_post"] else ""
    print("[human_node] Review the draft above.")
    print("Tip: press Enter for 'okay'. Type 'done' to finish. Or write feedback to iterate.\n")

    user_feedback = interrupt({
        "generated_post": latest,
        "message": "Your feedback [okay/done/custom]:"
    })

    # Normalize and route
    feedback_list = state.get("human_feedback", [])
    if isinstance(user_feedback, str):
        raw = user_feedback.strip()
        if not raw:
            raw = "okay"
        low = raw.lower()

        if low in DONE_SET:
            return Command(
                update={"human_feedback": feedback_list + ["Finalised"]},
                goto="end_node"
            )

        if low in OKAY_SET:
            feedback_list = feedback_list + ["okay"]
        else:
            feedback_list = feedback_list + [raw]
    else:
        feedback_list = feedback_list + [str(user_feedback)]

    return Command(update={"human_feedback": feedback_list}, goto="model")


def end_node(state: State) -> State:
    final_post = state["generated_post"][-1].content if state["generated_post"] else ""
    print("\n[end_node] All set.")
    print("Here is your final post:\n")
    print("===== FINAL POST =====")
    print(final_post)
    print("======================")
    print("\nFeedback trail:", state.get("human_feedback", []))
    return state


# -------------------------
# Graph
# -------------------------
graph = StateGraph(State)
graph.add_node("model", model)
graph.add_node("human_node", human_node)
graph.add_node("end_node", end_node)

graph.set_entry_point("model")
graph.add_edge(START, "model")
graph.add_edge("model", "human_node")
graph.set_finish_point("end_node")

checkpointer = MemorySaver()
app = graph.compile(checkpointer=checkpointer)

# -------------------------
# Run
# -------------------------
if __name__ == "__main__":
    thread_config = {"configurable": {"thread_id": str(uuid.uuid4())}}

    try:
        linkedin_topic = input("Enter your LinkedIn topic: ").strip()
    except KeyboardInterrupt:
        raise SystemExit("\nAborted.")

    if not linkedin_topic:
        raise SystemExit("Please provide a topic next time.")

    initial_state: State = {
        "linkedin_topic": linkedin_topic,
        "generated_post": [],
        "human_feedback": []
    }

    stream = app.stream(initial_state, config=thread_config)

    while True:
        try:
            chunk = next(stream)
            # An interrupt arrives as a special event
            if "__interrupt__" in chunk:
                # Interactive loop until we resume
                while True:
                    try:
                        user_feedback = input("Your feedback [okay/done/custom]: ").strip()
                    except KeyboardInterrupt:
                        user_feedback = "done"

                    if not user_feedback:
                        user_feedback = "okay"

                    if user_feedback.lower() in DONE_SET:
                        app.invoke(Command(resume="done"), config=thread_config)
                        break
                    else:
                        app.invoke(Command(resume=user_feedback), config=thread_config)
        except StopIteration:
            break

    print("Finished.")


Enter your LinkedIn topic: LLM agents

[model] Draft ready. Handing over for review...

----- DRAFT START -----
Large Language Model (LLM) agents are revolutionizing the way we work, and it's essential to understand their capabilities. With the ability to process and generate human-like language, LLM agents can automate tasks and provide insights that were previously unimaginable.

One key benefit of LLM agents is their ability to analyze vast amounts of data and identify patterns. For instance, a company like Netflix can use LLM agents to analyze user behavior and provide personalized recommendations, leading to increased user engagement and retention.

To get the most out of LLM agents, it's crucial to define clear objectives and provide high-quality training data. A practical example of this is a customer service chatbot that uses LLM agents to provide accurate and helpful responses to customer inquiries, resulting in improved customer satisfaction and reduced support costs.

If you

In [ ]:
from langgraph.graph import StateGraph, START, END, add_messages
from langgraph.types import Command, interrupt
from typing import TypedDict, Annotated, List
from langgraph.checkpoint.memory import MemorySaver
from langchain_openai.chat_models import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
import uuid

In [ ]:
class State(TypedDict):
    linkedin_topic: str
    generated_post: Annotated[List[str], add_messages]
    human_feedback: Annotated[List[str], add_messages]


In [ ]:
def model(state: State):
    """ Here, we're using the LLM to generate a first feedback human feedback
    incorporated on the initial prompt for the hypothesis."""

    hypothesis_prompt = state["linkedin_topic"]
    feedback = state["human_feedback"] if "human_feedback" in state else ["No Feedback yet"]


    # Define the prompt
    prompt = f"""

        LinkedIn Topic: {linkedin_topic}
        Human Feedback: {feedback[-1] if feedback else "No feedback yet"}

        Generate a structured and well-written LinkedIn post based on the given topic.

        Consider previous human feedback to refine the reponse.
    """

    response = llm.invoke([
        SystemMessage(content="You are an expert LinkedIn content writer"),
        HumanMessage(content=prompt)
    ])

    geneated_linkedin_post = response.content

    print(f"[model_node] Generated post:\n{geneated_linkedin_post}\n")

    return {
       "generated_post": [AIMessage(content=geneated_linkedin_post)] ,
       "human_feedback": feedback
    }

In [ ]:
def human_node(state: State):
    """Human Intervention node - loops back to model unless input is done"""

    print("\n [human_node] awaiting human feedback")

    generated_post = state["generated_post"]

    # Interrupt to get user feedback

    user_feedback = interrupt(
        {
            "generated_post": generated_post,
            "message": "Provide feedback or type 'done' to finish"
        }
    )

    print(f"[human_node] Received human feedback: {user_feedback}")


    # If user types an exit command, transition to END node
    if user_feedback.lower() in ["done", "quit", "exit", "q"]:
        return Command(update={"human_feedback": state["human_feedback"] + ["Finalised"]}, goto="end_node")


    # Otherwise, update feedback and return to model for re-generation
    return Command(update={"human_feedback": state["human_feedback"] + [user_feedback]}, goto="model")


In [ ]:
def end_node(state: State):
    """ Final node """
    print("\n[end_node] Process finished")
    print("Final Generated Post:", state["generated_post"][-1])
    print("Final Human Feedback", state["human_feedback"])
    return {"generated_post": state["generated_post"], "human_feedback": state["human_feedback"]}


# Buiding the Graph

In [ ]:
graph = StateGraph(State)
graph.add_node("model", model)
graph.add_node("human_node", human_node)
graph.add_node("end_node", end_node)

graph.set_entry_point("model")



# Define the flow

In [ ]:
graph.add_edge(START, "model")
graph.add_edge("model", "human_node")

graph.set_finish_point("end_node")

# Enable Interrupt mechanism

In [ ]:
checkpointer = MemorySaver()
app = graph.compile(checkpointer=checkpointer)

thread_config = {"configurable": {
    "thread_id": uuid.uuid4()
}}

linkedin_topic = input("Enter your LinkedIn topic: ")
initial_state = {
    "linkedin_topic": linkedin_topic,
    "generated_post": [],
    "human_feedback": []
}

# Initial call to stream the first output
stream = app.stream(initial_state, config=thread_config)
feedback_done = False

while True:
    try:
        chunk = next(stream)
        for node_id, value in chunk.items():
            if node_id == "__interrupt__":
                while True:
                    user_feedback = input("User: ")
                    if user_feedback.lower() in ["ok"]:
                        app.invoke(Command(resume="done"), config=thread_config)
                        feedback_done = True
                        break
                    else:
                        app.invoke(Command(resume=user_feedback), config=thread_config)
                if feedback_done:
                    break
    except StopIteration:
        break

print("Finished.")






Enter your LinkedIn topic: ai agents
[model] Generating content
[model_node] Generated post:
**The Rise of AI Agents: Revolutionizing Industries and Redefining Productivity**

As we continue to navigate the complexities of the digital age, Artificial Intelligence (AI) has emerged as a transformative force, revolutionizing the way we live, work, and interact. One of the most exciting developments in the AI landscape is the emergence of AI agents, designed to perform specific tasks, make decisions, and learn from their environment.

**What are AI Agents?**

AI agents are software programs that use machine learning algorithms to perceive their environment, make decisions, and take actions to achieve a specific goal. They can be simple or complex, depending on their design and purpose. From chatbots and virtual assistants to autonomous vehicles and smart home devices, AI agents are increasingly becoming an integral part of our daily lives.

**Key Benefits of AI Agents**

1. **Increased Eff